# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze data from a dataset described by a [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) Croissant schema using the `mlcroissant` library. All record sets, fields, and columns are referenced by their unique `@id`.

### Dataset Source
The dataset is described by the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Use the FAIR2 Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata  # Not a dictionary! Use attributes .name, .description, etc.
print(f"Dataset name: {meta.name}\n\nDescription: {meta.description}")

## 2. Data Overview
Review available record sets, their fields, and the unique `@id` for each entity.

In [ ]:
# List all record set @ids, field @ids and their descriptions
from pprint import pprint

print("\nAvailable Record Sets in the dataset:")
record_sets = []

# The mlcroissant metadata exposes record sets as a .record_sets property
for rs in meta.record_sets:
    print(f"- RecordSet @id: {rs.id} | Name: {rs.name} | Description: {getattr(rs, 'description', '')}")
    record_sets.append(rs.id)
    print("  Available fields and their @id:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id}\tName: {field.name}\tDataType: {field.data_type}")
    print()

# Show example records in the first record set
if record_sets:
    first_rs = record_sets[0]
    print(f"Sample record from record set '{first_rs}':")
    for i, record in enumerate(dataset.records(record_set=first_rs)):
        pprint(record)
        if i >= 1:
            break

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for further analysis. The `@id` of record sets and fields collected above are used as references.

In [ ]:
dataframes = {}
for rs_id in record_sets:
    print(f"Loading records from RecordSet @id: {rs_id} ...")
    records_list = list(dataset.records(record_set=rs_id))
    # If no records, warn and continue
    if not records_list:
        print(f"  No records found in {rs_id}")
        continue
    dataframes[rs_id] = pd.DataFrame(records_list)
    print(f"  Loaded DataFrame columns (field @ids): {dataframes[rs_id].columns.tolist()}")
    # Show top records
    display(dataframes[rs_id].head())

## 4. Exploratory Data Analysis (EDA)
We'll perform some EDA using field `@id`s for demonstration. For this, we pick a numeric field (e.g., coverage, peptide_count, or MW), filter data, normalize values, and optionally group by a categorical field. You can modify field selections below as per overview results.

In [ ]:
# Choose the first available record set for demo
if dataframes:
    from IPython.display import display
    # Choose the first record set for demonstration
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Using RecordSet @id: {record_set_id}")

    # List field @ids
    print("Available column (field @id) names:")
    print(df.columns.tolist())
    
    # Suppose 'cr:coverage' and 'cr:mw' are present as field @ids (adjust as needed)
    preferred_numeric_fields = [c for c in df.columns if c in [
        'cr:coverage', 'cr:mw', 'cr:peptide_count', 'cr:total_intensity', 'cr:psm_count'
    ]]
    if preferred_numeric_fields:
        numeric_field = preferred_numeric_fields[0]
        print(f"Using numeric field: {numeric_field}")
        try:
            # Convert to numeric if possible
            df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
            threshold = df[numeric_field].quantile(0.75)  # Use top quartile as filter for demonstration
            filtered_df = df[df[numeric_field] > threshold]
            print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
            display(filtered_df.head())

            # Normalization
            filtered_df[f"{numeric_field}_normalized"] = (
                filtered_df[numeric_field] - filtered_df[numeric_field].mean()
            ) / filtered_df[numeric_field].std()
            print(f"Normalized {numeric_field} for filtered records:")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            # Attempt to group by a categorical field (e.g., accession/type/sample)
            possible_group_fields = [c for c in df.columns if ('type' in c.lower() or 'sample' in c.lower() or 'group' in c.lower())]
            if possible_group_fields:
                group_field = possible_group_fields[0]
                print(f"Grouping by field @id: {group_field}")
                grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
                print(f"Grouped data by {group_field} (showing first rows):")
                display(grouped_df.head())
            else:
                print("No suitable group field found for grouping.")
        except Exception as e:
            print(f"Error during EDA: {e}")
    else:
        print("No obvious numeric fields found for EDA. Please refer to the columns list above.")
else:
    print("No dataframes available.")

## 5. Visualization
Visualize the distribution of a numeric field or the relationship between two relevant fields. Update field `@id` as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    if preferred_numeric_fields:
        plt.figure(figsize=(8, 5))
        sns.histplot(df[numeric_field].dropna(), kde=True, bins=25)
        plt.title(f"Distribution of field {numeric_field}")
        plt.xlabel(numeric_field)
        plt.show()

        # Scatter plot versus another numeric field if present
        if len(preferred_numeric_fields) > 1:
            y_field = preferred_numeric_fields[1]
            df[y_field] = pd.to_numeric(df[y_field], errors='coerce')
            plt.figure(figsize=(8, 5))
            sns.scatterplot(x=df[numeric_field], y=df[y_field])
            plt.xlabel(numeric_field)
            plt.ylabel(y_field)
            plt.title(f"Scatter of {numeric_field} vs {y_field}")
            plt.show()
    else:
        print("No numeric field for visualization.")

## 6. Conclusion
This notebook demonstrated how to programmatically load, explore, and process mass spectrometry protein data using the [mlcroissant](https://pypi.org/project/mlcroissant/) library, referencing all data entities by their Croissant `@id`s. You can adapt field and group selections for your analytic purposes using the overview above.